# Debug List Scraper Malang Posco
List-only. Tidak scrape artikel detail.

In [1]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.malangposco as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)
print('scrape_list signature:', inspect.signature(scraper.scrape_list))


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/malangposco.py
scrape_list signature: (cutoff, max_pages=200)


In [5]:
# ```python
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = "" if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def ensure_debug_columns(df):
    df = df.copy()

    if "page_num" not in df.columns:
        df["page_num"] = pd.NA

    if "published_date" not in df.columns:
        df["published_date"] = None

    df["published_dt"] = df["published_date"].apply(parse_date)

    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get("page_num")
        page_text = "---" if pd.isna(page) else f"{int(page):03d}"

        date_text = row.get("published_date")
        date_text = "None" if pd.isna(date_text) else str(date_text)

        print(
            f"page={page_text} | "
            f"date={date_text} | "
            f"title={short_title(row.get('title'))}"
        )


def audit_list(df):
    df = ensure_debug_columns(df)

    print("total rows:", len(df))

    if len(df) == 0:
        print("No rows found.")
        return df

    print(
        "first page:",
        df["page_num"].dropna().min()
        if df["page_num"].notna().any()
        else None,
    )

    print(
        "last page:",
        df["page_num"].dropna().max()
        if df["page_num"].notna().any()
        else None,
    )

    print("newest date:", df["published_dt"].max())
    print("oldest date:", df["published_dt"].min())
    print("cutoff date:", cutoff)
    print("reached cutoff:", bool((df["published_dt"].dropna() < cutoff).any()))
    print("null parsed dates:", int(df["published_dt"].isna().sum()))

    print("\nrows per month:")
    display(
        df.assign(month=df["published_dt"].dt.to_period("M").astype(str))
        .groupby("month", dropna=False)
        .size()
        .reset_index(name="count")
    )

    print("\nrows per page:")
    display(
        df.groupby("page_num", dropna=False)
        .agg(
            rows=("url", "count"),
            newest=("published_dt", "max"),
            oldest=("published_dt", "min"),
        )
        .reset_index()
        .tail(60)
    )

    return df
# ```

In [6]:
list_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)


Malang Posco list max_pages=200, cutoff=2026-02-28
Scraping Malang Posco page 1
Scraping Malang Posco page 2
Scraping Malang Posco page 3
Scraping Malang Posco page 4
Scraping Malang Posco page 5
Scraping Malang Posco page 6
Scraping Malang Posco page 7
Scraping Malang Posco page 8
Scraping Malang Posco page 9
Scraping Malang Posco page 10
Scraping Malang Posco page 11
Scraping Malang Posco page 12
Scraping Malang Posco page 13
Scraping Malang Posco page 14
Scraping Malang Posco page 15
Scraping Malang Posco page 16
Scraping Malang Posco page 17
Scraping Malang Posco page 18
Scraping Malang Posco page 19
Scraping Malang Posco page 20
Scraping Malang Posco page 21
Scraping Malang Posco page 22
Scraping Malang Posco page 23
Scraping Malang Posco page 24
Scraping Malang Posco page 25
Scraping Malang Posco page 26
Scraping Malang Posco page 27
Scraping Malang Posco page 28
Scraping Malang Posco page 29
Scraping Malang Posco page 30
Scraping Malang Posco page 31
Scraping Malang Posco page 3

In [7]:
list_df = audit_list(list_df)


total rows: 582
first page: 1
last page: 97
newest date: 2026-06-28 23:32:20
oldest date: 2026-02-28 12:49:47
cutoff date: 2026-02-28 08:37:58.969097
reached cutoff: False
null parsed dates: 0

rows per month:


,month,count
0,2026-02,2
1,2026-03,194
2,2026-04,147
3,2026-05,116
4,2026-06,123



rows per page:


,page_num,rows,newest,oldest
37,38,6,2026-05-06 13:40:14,2026-05-05 13:48:01
38,39,6,2026-05-05 13:14:17,2026-05-04 16:29:20
39,40,6,2026-05-03 21:36:33,2026-04-29 20:48:51
40,41,6,2026-04-29 20:47:34,2026-04-28 20:00:50
41,42,6,2026-04-28 19:58:37,2026-04-27 17:30:41
42,43,6,2026-04-27 15:33:12,2026-04-26 21:10:10
43,44,6,2026-04-26 21:08:31,2026-04-25 12:45:17
44,45,6,2026-04-25 10:22:27,2026-04-23 21:15:37
45,46,6,2026-04-23 20:40:02,2026-04-23 16:51:33
46,47,6,2026-04-23 08:00:00,2026-04-22 19:37:50


In [8]:
list_df.sort_values('published_dt')[['page_num', 'published_date', 'published_dt', 'title', 'url']].head(50)


,page_num,published_date,published_dt,title,url
581,97,2026-02-28T12:49:47+07:00,2026-02-28 12:49:47,"Museum Tragedi Kanjuruhan Diisi Barang Milik Korban, Dikelola YKTK dan Dispora",https://malangposcomedia.id/museum-tragedi-kanjuruhan-diisi-barang-milik-korban-dikelola-yktk-dan-dispora/
580,97,2026-02-28T18:50:02+07:00,2026-02-28 18:50:02,"DPC PERADI Kabupaten Malang Gelar RAC, Optimis Fikri Assegaf Jadi Ketum DPN",https://malangposcomedia.id/dpc-peradi-kabupaten-malang-gelar-rac-optimis-fikri-assegaf-jadi-ketum-dpn/
579,97,2026-03-01T13:31:33+07:00,2026-03-01 13:31:33,"Jual Motor Curian di Marketplace, Diringkus Polisi",https://malangposcomedia.id/jual-motor-curian-di-marketplace-diringkus-polisi/
578,97,2026-03-01T15:38:17+07:00,2026-03-01 15:38:17,"Sempat Dicari Tim SAR, Pria Asal Ngajum Hanyut di Sungai, Ternyata Pulang Naik Ojek",https://malangposcomedia.id/sempat-dicari-tim-sar-pria-asal-ngajum-hanyut-di-sungai-ternyata-pulang-naik-ojek/
577,97,2026-03-01T17:24:58+07:00,2026-03-01 17:24:58,Tembok Pembatas Perumahan di Desa Asrikaton Longsor,https://malangposcomedia.id/tembok-pembatas-perumahan-di-desa-asrikaton-longsor/
576,97,2026-03-01T20:46:59+07:00,2026-03-01 20:46:59,"­UPT Puskesmas Wagir; Inovasi KECE BINGITS, Perkuat Investigasi Kontak dan Terapi Pencegahan TBC",https://malangposcomedia.id/upt-puskesmas-wagir-inovasi-kece-bingits-perkuat-investigasi-kontak-dan-terapi-pencegahan-tbc/
575,96,2026-03-01T20:47:42+07:00,2026-03-01 20:47:42,"PIKAT CINTA, Pendekatan Kearifan Lokal untuk Cegah Kematian Ibu dan Bayi",https://malangposcomedia.id/pikat-cinta-pendekatan-kearifan-lokal-untuk-cegah-kematian-ibu-dan-bayi/
574,96,2026-03-01T21:07:53+07:00,2026-03-01 21:07:53,Masyarakat Antusias Ikuti Mudik Gratis,https://malangposcomedia.id/masyarakat-antusias-ikuti-mudik-gratis/
573,96,2026-03-01T23:24:55+07:00,2026-03-01 23:24:55,"Orchid Koz Hadir di Kota Malang, Investasi Jangka Panjang Hanya Bayar Sekali di Depan",https://malangposcomedia.id/orchid-koz-hadir-di-kota-malang-investasi-jangka-panjang-hanya-bayar-sekali-di-depan/
572,96,2026-03-02T15:03:36+07:00,2026-03-02 15:03:36,"Suara Dentuman Keras di Dubai, Penentuan Juara Tahfidz Quran Asal Malang Tertunda",https://malangposcomedia.id/suara-dentuman-keras-di-dubai-penentuan-juara-tahfidz-quran-asal-malang-tertunda/


In [9]:
output_path = PROJECT_ROOT / 'csv' / 'malangposco_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/malangposco_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'malangposco_article_debug.csv'

article_rows = []
sample_rows = list_df.head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
